In [2]:
from datasets import load_dataset

#ds will become a DatasetDict. That usually means it contains splits like: train test
ds = load_dataset("timdettmers/openassistant-guanaco")

print (ds)

print(ds["train"][0]["text"][:500])

Repo card metadata block was not found. Setting CardData to empty.


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 9846
    })
    test: Dataset({
        features: ['text'],
        num_rows: 518
    })
})
### Human: Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.### Assistant: "Monopsony" refers to a market structure where there is only one buyer for a particular good or service. In economics, this term is particularly relevant in the labor market, where a monopsony employer has significant power over the wages and working conditions of their employees


In [ ]:
from transformers import AutoTokenizer

MODEL = "meta-llama/Llama-3.2-1B-Instruct"

tok = AutoTokenizer.from_pretained (MODEL) #convert text into ids

#if the tokenizer doesnt have atoken for padding we speicfy one in case differents elemtns have diff sizes
#Used to make examples in a batch the same length
if tok.pad_token is None:
  tok.pad_token = tok.eos_token

In [ ]:
# It takes one dataset row as input.
# A dataset row looks like this:
# {
#     "text": "### Human: ... ### Assistant: ..."
# }
def to_chat(example):

  text = example["text"]

  msgs = []

  for block in text.split('### '):
    block = block.strip()

    if block.startswith("Human:"):
      msgs.append({
          "role": "user",
          "content": block[len("Human:"):].strip()
      })
    elif block.startswith('Assistant:'):
          msgs.append({
        "role": "assistant",
        "content": block[len("Assistant:"):].strip()
    })
  if not msgs or msgs[-1]["role"] != "assistant":
    return {"text": None}
  #converts it into the special Llama chat format with special tokens. tokenize=False means: Return text, not token IDs yet
  return {"text": tok.apply_chat_template(msgs, tokenize=False)}


  #take the training split from the dataset
  #.map(to_chat) This means: Run to_chat(example) on every single row in the training dataset.
  #filter(lambda e: e["text"] is not None) : This removes the bad rows.For each example e, keep it only if its "text" is not None
  train_ds = ds[train].map(to_chat).filter(lambda e: e["text"] is not None)
  eval_ds = ds[text].map(to_chat).filter(lambda e: e["text"] is not None)




In [1]:
!pip install peft==0.12.0 trl==0.9.6 datasets==2.21.0

In [ ]:
import torch, wandb
from prepare_data import load_formatted_data

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training,
    get_peft_model,
)

from trl import SFTTrainer, SFTConfig

MODEL = "meta-llama/Llama-3.2-1B-Instruct" #the original Llama model we start from

OUT = "models/day09_llama32_1b_oa_lora" #After training, we save the LoRA adapter here.

def main():

  wandb.init(project="ai-trainee", name="day09-qlora-llama32-1b")

  tok = AutoTokenizer.from_pretrained(MODEL)

  #If there is no pad token, use the end-of-sequence token as padding.
  if tok.pad_token is None:
    tok.pad_token = tok.eos_token

#When we load the base Llama model, load it in 4-bit instead of full precision
#What is 4-bit again? Normally model weights might be stored as fp16. fp16 = 16 bits per number QLoRA stores the frozen base model weights in 4-bit.
#4-bit = 4 bits per number So memory goes way down. Very rough idea: fp16 model: bigger VRAM usage 4-bit model: much smaller VRAM usage
  bnb = BitsAndBytesConfig( #creates a config object
      load_in_4bit=True,
      bnb_4bit_quant_type="nf4",
      bnb_4bit_compute_dtype=torch.float16, #Store the base model in 4-bit, but do computations using fp16.
      bnb_4bit_use_double_quant=True,
  )

  model = AutoModelForCausalLM.from_pretrained( #load model auto
      MODEL, #Load this specific model from the Hub.
      quantization_config=bnb, #Use those settings while loading the model.
      device_map="auto",
  )
  #Prepare the compressed base model so LoRA adapters can be trained on top of it
  #4-bit base model needs special preparation before LoRA training.
  model = prepare_model_for_kbit_training(model)

  model.config.use_cache = False

  #creates the settings for the LoRA adapter

  peft_cfg = LoraConfig(
    r=16, #size/capacity
    lora_alpha=32, #controls how strongly the adapter affects the model
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM", #This tells PEFT what kind of model/task we are fine-tuning. CAUSAL_LM means: Causal Language Modeling
    target_modules=[ #Which parts of the Llama model should receive LoRA adapters?These are part of the attention mechanism. Attention is the part that helps the model decide: Which previous words matter for the next word? So adding LoRA here helps change how the model attends to information.
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)



train_ds, eval_ds = load_formatted_data(tok)

#When training starts, use these settings.
cfg = SFTConfig(
    output_dir=OUT,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    bf16=False,
    fp16=True,
    gradient_checkpointing=True,
    max_seq_length=1024,
    dataset_text_field="text",
    packing=True,
    report_to="wandb",
)
#SFTTrainer connects everything together
trainer = SFTTrainer(
    model=model,
    tokenizer=tok,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_cfg, #Attach LoRA adapters using this configuration.
    args=cfg,
)
trainer.train()
trainer.save_model(OUT)